# Kubeflow Trainer: Minimal Torch Job (kind / CPU) — OTel Instrumented

Smallest possible `torch-distributed` TrainJob with **end-to-end trace propagation**.

Trace flow:
```
notebook (SDK)
  └─ trainer.submit          ← CLIENT span (TrainerClient.train)
       └─ [context injected as TRACEPARENT env var into pods]
            └─ training.worker   ← SERVER span (each pod, linked to parent)
                 └─ training.epoch  ← INTERNAL span (per epoch)
```

In [1]:
import sys

import kubeflow
print(kubeflow.__file__)

/Users/sujalshah/Desktop/code/opensource/kubeflow/sdk/kubeflow/__init__.py


### OTEL Setup
Build the tracer provider and pass it into the TrainerClient.

In [2]:
def build_tracer_provider(
    service_name: str = "kubeflow-trainer-example",
    backend: str = "jaeger",   # "jaeger" | "otlp-collector" | "console"
    endpoint: str = "http://localhost:4317",
    headers: dict = None,
):
    """
    Build and return a TracerProvider configured for the chosen backend.
    """

    from opentelemetry.propagate import set_global_textmap
    from opentelemetry.sdk.resources import Resource
    from opentelemetry.sdk.trace import TracerProvider
    from opentelemetry.sdk.trace.export import BatchSpanProcessor
    from opentelemetry.trace.propagation.tracecontext import TraceContextTextMapPropagator

    resource = Resource.create({
        "service.name": service_name,
        "service.version": "0.1.0",
    })

    provider = TracerProvider(resource=resource)

    if backend in ("jaeger", "otlp-collector"):
        from opentelemetry.exporter.otlp.proto.grpc.trace_exporter import OTLPSpanExporter
        exporter = OTLPSpanExporter(
            endpoint=endpoint,
            headers=headers or {},
            insecure=True,
        )
    elif backend == "console":
        from opentelemetry.sdk.trace.export import ConsoleSpanExporter
        exporter = ConsoleSpanExporter()
    else:
        raise ValueError(f"Unknown backend '{backend}'.")

    provider.add_span_processor(BatchSpanProcessor(exporter))

    # W3C TraceContext is the wire format — sets traceparent / tracestate headers.
    set_global_textmap(TraceContextTextMapPropagator())

    return provider

## Training function

Runs on every worker pod inside the cluster.

**Trace context** is picked up from the `TRACEPARENT` / `TRACESTATE` env vars that the SDK injects via `podTemplateOverrides`. Each worker starts a `SERVER` span that is a **child of the SDK's `trainer.submit` span**, so everything appears in one trace in Jaeger.

In [3]:
def train_fn():
    import os
    import time
    import torch
    import torch.distributed as dist
    import subprocess, sys
    try:
        import opentelemetry
    except ImportError:
        subprocess.check_call([
            sys.executable, "-m", "pip", "install", "-q",
            "--break-system-packages",
            "opentelemetry-sdk",
            "opentelemetry-exporter-otlp-proto-grpc",
        ])


    # ── 1. Restore trace context that the SDK injected as env vars ────────────
    # The SDK calls opentelemetry.propagate.inject(carrier) and then passes
    # {'TRACEPARENT': '...', 'TRACESTATE': '...'}.
    # W3C spec uses lowercase header names; env vars are uppercase — normalise here.
    from opentelemetry import trace
    from opentelemetry.propagate import extract, set_global_textmap
    from opentelemetry.sdk.resources import Resource
    from opentelemetry.sdk.trace import TracerProvider
    from opentelemetry.sdk.trace.export import BatchSpanProcessor
    from opentelemetry.trace import SpanKind, Link
    from opentelemetry.trace.propagation.tracecontext import TraceContextTextMapPropagator

    # Must set the global propagator before calling extract().
    set_global_textmap(TraceContextTextMapPropagator())

    # Read W3C headers from env vars (SDK injects TRACEPARENT / TRACESTATE).
    carrier = {
        "traceparent": os.environ.get("TRACEPARENT", ""),
        "tracestate":  os.environ.get("TRACESTATE",  ""),
    }
    print("TRACEPARENT present:", "TRACEPARENT" in os.environ)
    print("TRACEPARENT value:", repr(os.environ.get("TRACEPARENT")))
    print("TRACESTATE present:", "TRACESTATE" in os.environ)
    print("TRACESTATE value:", repr(os.environ.get("TRACESTATE")))
    parent_ctx = extract(carrier)   # returns a valid context even if env vars are empty
    span_context = trace.get_current_span(parent_ctx).get_span_context()
    # ── 2. Configure the in-pod tracer provider ───────────────────────────────
    # Reads the OTEL collector endpoint from env; falls back to a sane default.
    # You can inject OTEL_EXPORTER_OTLP_ENDPOINT via podTemplateOverrides too.
    otlp_endpoint = os.environ.get(
        "OTEL_EXPORTER_OTLP_ENDPOINT",
        "http://jaeger.default.svc.cluster.local:4317"  # adjust namespace
    )

    rank = int(os.environ.get("RANK", "0"))

    resource = Resource.create({
        "service.name":    "kubeflow-trainer-worker",
        "service.version": "0.1.0",
        "k8s.pod.name":    os.environ.get("POD_NAME", f"worker-{rank}"),
        "training.rank":   rank,
    })

    provider = TracerProvider(resource=resource)

    try:
        from opentelemetry.exporter.otlp.proto.grpc.trace_exporter import OTLPSpanExporter
        exporter = OTLPSpanExporter(endpoint=otlp_endpoint, insecure=True)
    except ImportError:
        # Fallback: print to stdout so spans are visible in pod logs.
        from opentelemetry.sdk.trace.export import ConsoleSpanExporter
        exporter = ConsoleSpanExporter()

    provider.add_span_processor(BatchSpanProcessor(exporter))
    tracer = provider.get_tracer("kubeflow.trainer.worker")

    # ── 3. Root worker span — child of the SDK's trainer.submit span ──────────
    with tracer.start_as_current_span(
        "training.worker",
        kind=SpanKind.CONSUMER,
        context=parent_ctx
        # links=[Link(span_context)]   # <-- link, not parent
    )as worker_span:
        worker_span.set_attribute("training.rank",       rank)
        worker_span.set_attribute("training.backend",    "gloo")
        worker_span.set_attribute("training.num_epochs", 3)

        dist.init_process_group(backend="gloo")
        world_size = dist.get_world_size()
        worker_span.set_attribute("training.world_size", world_size)
        print(f"[rank {rank}/{world_size}] init done")

        for epoch in range(1, 4):
            # ── 4. Per-epoch child span ───────────────────────────────────────
            with tracer.start_as_current_span(
                "training.epoch",
                kind=SpanKind.INTERNAL,
            ) as epoch_span:
                epoch_span.set_attribute("training.epoch",      epoch)
                epoch_span.set_attribute("training.rank",       rank)

                t    = torch.randn(128, 128)
                loss = (t @ t.T).mean().item()
                time.sleep(0.1)

                epoch_span.set_attribute("training.loss", round(loss, 6))

            dist.barrier()
            print(f"[rank {rank}] epoch {epoch}/3  loss={loss:.4f}")

        dist.barrier()
        if rank == 0:
            print("Training complete")

        dist.destroy_process_group()

    # Flush remaining spans before the process exits.
    provider.force_flush()

## Create client & pick runtime

In [4]:
from kubeflow.trainer import TrainerClient, CustomTrainer
from opentelemetry import trace

provider = build_tracer_provider(
    service_name="kubeflow-trainer-example",
    backend="jaeger",            # swap to "jaeger" once your collector is running
    endpoint="http://localhost:4317",
)
trace.set_tracer_provider(provider) 

client = TrainerClient(tracer_provider=provider)

torch_runtime = None
for runtime in client.list_runtimes():
    print(runtime)
    if runtime.name == "torch-distributed":
        torch_runtime = runtime

assert torch_runtime is not None, "torch-distributed runtime not found"

Runtime(name='deepspeed-distributed', trainer=RuntimeTrainer(trainer_type=<TrainerType.CUSTOM_TRAINER: 'CustomTrainer'>, framework='deepspeed', image='ghcr.io/kubeflow/trainer/deepspeed-runtime:latest', num_nodes=1, device='Unknown', device_count='1'), pretrained_model=None)
Runtime(name='jax-distributed', trainer=RuntimeTrainer(trainer_type=<TrainerType.CUSTOM_TRAINER: 'CustomTrainer'>, framework='jax', image='nvcr.io/nvidia/jax:25.10-py3', num_nodes=1, device='Unknown', device_count='Unknown'), pretrained_model=None)
Runtime(name='mlx-distributed', trainer=RuntimeTrainer(trainer_type=<TrainerType.CUSTOM_TRAINER: 'CustomTrainer'>, framework='mlx', image='ghcr.io/kubeflow/trainer/mlx-runtime:latest', num_nodes=1, device='Unknown', device_count='1'), pretrained_model=None)
Runtime(name='torch-distributed', trainer=RuntimeTrainer(trainer_type=<TrainerType.CUSTOM_TRAINER: 'CustomTrainer'>, framework='torch', image='pytorch/pytorch:2.10.0-cuda12.8-cudnn9-runtime', num_nodes=1, device='Unkn

## Submit

The SDK's `train()` method:
1. Opens a `trainer.submit` CLIENT span.
2. Calls `inject(carrier)` → fills `{'traceparent': '...', 'tracestate': '...'}`.
3. Passes the carrier as env vars via `podTemplateOverrides` so every pod inherits the trace context.

If you also want to forward the OTLP endpoint so pods know where to export:

```python
from kubeflow.trainer import models

otel_override = models.TrainerV1alpha1PodTemplateOverride(
    containers=[
        models.TrainerV1alpha1ContainerOverride(
            name="trainer",
            env=[
                models.IoK8sApiCoreV1EnvVar(
                    name="OTEL_EXPORTER_OTLP_ENDPOINT",
                    value="http://jaeger.monitoring.svc.cluster.local:4317",
                ),
                models.IoK8sApiCoreV1EnvVar(
                    name="POD_NAME",
                    value_from=models.IoK8sApiCoreV1EnvVarSource(
                        field_ref=models.IoK8sApiCoreV1ObjectFieldSelector(field_path="metadata.name")
                    ),
                ),
            ],
        )
    ]
)
```

Then pass `pod_template_overrides=[otel_override]` to `client.train(...)`.

In [5]:
job_name = client.train(
    trainer=CustomTrainer(
        func=train_fn,
        num_nodes=2,
        resources_per_node={"cpu": 1},
    ),
    runtime=torch_runtime,
)
print(f"Submitted: {job_name}")

DEBUG carrier: {'traceparent': '00-f93dae383a4616fdbdc5cff86509485f-532fccc4cab32421-01'}
DEBUG span valid: True
DEBUG span id: 0x532fccc4cab32421
DEBUG [IoK8sApiCoreV1EnvVar(name='TRACEPARENT', value='00-f93dae383a4616fdbdc5cff86509485f-532fccc4cab32421-01', value_from=None)]

Submitted: b3d496f5ea95


## Wait for Running, then stream logs

In [6]:
job = client.get_job(job_name)
print(f"Job: {job.name}  Status: {job.status}")
for step in job.steps:
    print(f"  step={step.name}  status={step.status}")

Job: b3d496f5ea95  Status: Running
  step=node-0  status=Running
  step=node-1  status=Running


In [7]:
# Logs from node-0 (rank 0). Change step="node-1" for the other worker.
for line in client.get_job_logs(job_name, follow=True, step="node-0"):
    print(line, end="")

## Cleanup

In [8]:
client.delete_job(job_name)